# 🔐 Day 01a: Critical Section Problem & Race Conditions

---

## 🎯 What You'll Master Today
- Race conditions — why they happen
- Critical section requirements: mutual exclusion, progress, bounded waiting
- Peterson's solution, hardware solutions (test-and-set, CAS)

---

## 🎭 The Analogy

**Race condition** = two people editing the same Google Doc cell simultaneously without seeing each other's changes. One overwrite wins, the other is lost.

**Critical section** = a single-person bathroom. Only one person at a time. Lock the door (mutual exclusion), don't stay forever (progress), everyone gets a turn (bounded waiting).

---

```
╔══════════════════════════════════════════════════════════╗
║        CRITICAL SECTION REQUIREMENTS                     ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  1. Mutual Exclusion — only 1 process in CS at a time   ║
║  2. Progress — if CS is free, someone should enter       ║
║  3. Bounded Waiting — no starvation (finite wait)        ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
```

In [ ]:
# ============================================
# 1. Race Condition — The Problem
# ============================================
import threading

# Shared bank account
balance = 1000

def withdraw(amount, name):
    global balance
    temp = balance       # READ
    # Context switch can happen here!
    temp -= amount       # MODIFY
    balance = temp       # WRITE
    print(f"  {name} withdrew {amount}. Balance: {balance}")

# Both try to withdraw 800 from balance of 1000
# Without synchronization, both might see balance=1000!
balance = 1000
t1 = threading.Thread(target=withdraw, args=(800, "Alice"))
t2 = threading.Thread(target=withdraw, args=(800, "Bob"))
t1.start(); t2.start()
t1.join(); t2.join()
print(f"  Final balance: {balance} (should never be negative!)")

In [ ]:
# ============================================
# 2. Peterson's Solution (Two-Process)
# ============================================

class PetersonLock:
    """Classic software solution for 2 processes."""
    def __init__(self):
        self.flag = [False, False]  # intention to enter CS
        self.turn = 0               # whose turn if both want in
    
    def acquire(self, pid: int):
        other = 1 - pid
        self.flag[pid] = True   # I want to enter
        self.turn = other       # Give priority to other (courtesy)
        
        # Busy wait while other wants in AND it's their turn
        while self.flag[other] and self.turn == other:
            pass  # spin
    
    def release(self, pid: int):
        self.flag[pid] = False


# Demo (conceptual — Python GIL makes this hard to truly demo)
lock = PetersonLock()
shared = {"counter": 0}

def peterson_worker(pid, n):
    for _ in range(n):
        lock.acquire(pid)
        # --- CRITICAL SECTION ---
        shared["counter"] += 1
        # --- END CS ---
        lock.release(pid)

t0 = threading.Thread(target=peterson_worker, args=(0, 100_000))
t1 = threading.Thread(target=peterson_worker, args=(1, 100_000))
t0.start(); t1.start()
t0.join(); t1.join()
print(f"  Counter: {shared['counter']} (expected: 200,000)")

In [ ]:
# ============================================
# 3. Hardware Solutions — Test-and-Set, CAS
# ============================================

# Test-and-Set (atomic operation — hardware guaranteed)
# In real hardware this is a single atomic instruction

class TestAndSetLock:
    """Simulates test_and_set atomic instruction."""
    def __init__(self):
        self._lock_var = False
        self._real_lock = threading.Lock()  # simulates atomicity
    
    def test_and_set(self) -> bool:
        """Atomically: old = lock; lock = True; return old"""
        with self._real_lock:
            old = self._lock_var
            self._lock_var = True
            return old
    
    def acquire(self):
        while self.test_and_set():  # spin while locked
            pass
    
    def release(self):
        self._lock_var = False


# Compare-And-Swap (CAS) — more flexible
class CASLock:
    def __init__(self):
        self._val = 0
        self._real_lock = threading.Lock()
    
    def compare_and_swap(self, expected, new_val) -> bool:
        """Atomically: if val == expected: val = new; return True"""
        with self._real_lock:
            if self._val == expected:
                self._val = new_val
                return True
            return False
    
    def acquire(self):
        while not self.compare_and_swap(0, 1):
            pass
    
    def release(self):
        self._val = 0


print("Test-and-Set and CAS are HARDWARE atomic instructions.")
print("They guarantee atomicity at the CPU level (single bus cycle).")
print("Python threading.Lock() uses these under the hood!")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | What is a race condition? | Two+ processes access shared data concurrently, outcome depends on timing |
| 2 | Three requirements for CS solution? | Mutual exclusion, progress, bounded waiting |
| 3 | Peterson's limitations? | Only works for 2 processes, busy waiting, requires strict memory ordering |
| 4 | Test-and-Set vs CAS? | TAS always sets to true. CAS only sets if current matches expected (more flexible) |
| 5 | Spinlock vs blocking lock? | Spinlock busy-waits (good for short CS). Blocking puts thread to sleep (good for long CS) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Race condition = shared data + concurrent access     │
## │  • CS solution needs: mutex + progress + bounded wait    │
## │  • Peterson's = classic 2-process software solution      │
## │  • Hardware: TAS and CAS are atomic CPU instructions    │
## │  • Real systems use OS-provided locks (futex on Linux)  │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Mutex & Semaphore** — the synchronization toolkit